---

# JOINTS: audit validasi setelah v13

*Model dibekukan; validasi temporal dan pergeseran distribusi diperiksa terpisah.*

---

## Tim

Notebook eksperimen internal. Informasi tim mengikuti notebook utama.

---

## Daftar Isi

1. [Pendahuluan](#intro)
2. [Inisialisasi](#init)
3. [Data dan preprocessing](#data)
4. [Audit model tetap](#audit)

# 1. Pendahuluan <a id="intro"></a>

Pipeline B1 v13 digunakan tanpa pencarian hyperparameter. Empat cutoff bulanan memakai film training yang D10-nya selesai sebelum cutoff. Holdout 51 film dengan distribusi fitur mendekati test merupakan stress test non-temporal. Semua model tetap dikelompokkan per judul dasar. D1 rekonstruksi dan kalender tetap seperti v13; fitur lintas-film menggunakan jendela resmi yang tersedia dalam paket. Oleh sebab itu ini label-purged backtest, bukan simulasi real-time sempurna. September pernah dievaluasi sebelumnya dan bukan holdout baru yang belum tersentuh.

Tiga kandidat tetap: blend v13 (0.2 LGB, 0.5 Fast, 0.3 TabM), blend tanpa TabM (2/7 LGB, 5/7 Fast), serta CDF mixture dengan bobot v13. Tidak ada pemilihan bobot berdasarkan hasil run ini. Perbaikan CDF pada OOF sebelumnya dibayar dengan kerugian growth, sehingga seluruh segmen wajib dilaporkan.

Notebook ini tidak menulis submission atau memilih model final otomatis.

# 2. Inisialisasi <a id="init"></a>

Jalankan pada Kaggle T4 dengan data kompetisi dan HF_TOKEN. Instalasi serta implementasi model mengikuti v13.

In [ ]:
%pip install -q lightgbm==4.6.0 tabpfn==9.0.0 tabm==0.0.3 rtdl_num_embeddings==0.0.12 huggingface_hub==0.34.4 safetensors==0.8.0 scikit-learn==1.6.1 pandas==2.2.3 scipy==1.15.2 matplotlib==3.10.0 seaborn==0.13.2

In [ ]:
import os
import random
import time
import pickle
import io
import zlib
import itertools
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import torch
from IPython.display import display
from huggingface_hub import hf_hub_download
from safetensors import safe_open
from safetensors.numpy import save_file
from scipy.stats import ks_2samp
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.preprocessing import QuantileTransformer
from tabm import TabM
from rtdl_num_embeddings import PiecewiseLinearEmbeddings, compute_bins
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion
try:
    from kaggle_secrets import UserSecretsClient
except ImportError:
    UserSecretsClient = None

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
PAL = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#4a3aa7', '#e34948']
sns.set_theme(style='whitegrid', palette=PAL, rc={'axes.spines.top': False, 'axes.spines.right': False})
import json
assert Path("/kaggle/input").exists(), "Training audit hanya dijalankan di Kaggle"

In [ ]:
def seed_everything(seed: int = 2026):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(2026)

In [ ]:
class Settings:
    SEED       = 2026
    _ON_KAGGLE = Path('/kaggle/input').exists()

    DATA_DIR   = (
        next(Path('/kaggle/input').rglob('train.csv')).parent
        if _ON_KAGGLE else Path('../data')
    )
    OUTPUT_DIR = Path('/kaggle/working') if _ON_KAGGLE else Path('../outputs/v13')
    FIG_DIR    = OUTPUT_DIR / 'figures'

    TRAIN_END  = pd.Timestamp('2025-09-30')
    OUTAGE     = pd.to_datetime(['2025-06-07', '2025-06-09', '2025-06-10', '2025-06-13', '2025-06-16'])
    D1_FRAC    = 0.5
    MIN_NC     = 25
    USE_LIMITED = True

    DOW_PROF   = np.array([0.811, 0.789, 0.811, 0.783, 0.848, 1.290, 1.282])
    HOL_LEVEL  = 1.29
    SCHOOL_WD  = 1.15
    QS         = np.round(np.arange(0.05, 1.0, 0.05), 2)
    TAB_QS     = np.round(np.arange(0.02, 1.0, 0.02), 2)
    ZERO_EPS   = 0.02
    HURDLE_W   = 0.75
    LAMBDA     = 0.5
    LEB_RHO    = 0.5
    LEB_DAY1   = 0.75
    TW_BINS    = [0, 5, 20, 50, 100, 200, 500, 1e9]

    N_FOLDS    = 5
    SEEDS      = [2026, 2027, 2028, 2029, 2030]
    LGB_PARAMS = dict(objective='l1', learning_rate=0.03, num_leaves=63, min_child_samples=100,
                      feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
                      n_estimators=800, deterministic=True, force_row_wise=True, n_jobs=4, verbose=-1)

    REL_WIN      = 14
    QUIET_FRAC   = 0.4
    REL_GUARD    = 0.003
    # TabPFN candidates, at most one ships: 'tabpfn25q', 'fast35'; ('tabpfn_v2',) = revision 11 Jun 2025, before the cutoff
    FM_LIST      = ('tabpfn25q', 'fast35')
    FM_EST       = 8
    FAST_REPO    = 'Prior-Labs/tabpfn_3_5'                      # 334.2 MB float32, stored as matrix-FP16 (167.2 MB)
    FAST_FILE    = 'tabpfn-v3.5-fast-20260909.safetensors'
    FAST_REV     = '06bf2ba35c80a92a3b9abb436b99cf49e7a0365e'
    FAST_SHA256  = '14e0b81285bca7a6d1ef121169f27f651f76dff49cad6c0b8779d5c836018550'
    FAST16_FILE  = 'tabpfn-v3.5-fast-matrix-fp16.safetensors'
    SIZE_MARGIN_MB = 1.5
    TP25_REPO    = 'Prior-Labs/tabpfn_2_5'                      # 40.8 MB
    TP25_FILE    = 'tabpfn-v2.5-regressor-v2.5_quantiles.ckpt'
    TP25_REV     = '6c45f3a6d0d07c6c5f62572e04a0c2929de91b8b'
    TP25_SHA256  = '6dd4dbcdd5b991fde435ba5d59e4f6d170ec52110f5bb313ae8e972b88898044'
    USE_TABM     = True
    TABM_K       = 32
    TABM_BINS    = 48
    TABM_DEMB    = 16
    TABM_LR      = 2e-3
    TABM_WD      = 3e-4
    TABM_BATCH   = 256
    TABM_EPOCHS  = 40
    TABM_PATIENCE = 5
    TABPFN_REPO  = 'Prior-Labs/TabPFN-v2-reg'                   # 44.4 MB
    TABPFN_FILE  = 'tabpfn-v2-regressor.ckpt'
    TABPFN_REV   = '213f8e38ec399a2a385fa46cab6f22b95cd90de8'
    TABPFN_SHA256 = '2ab5a07d5c41dfe6db9aa7ae106fc6de898326c2765be66505a07e2868c10736'
    STRICT       = True
    MAX_WEIGHTS_MB = 200
    KAPPA        = 0.5
    KAPPA_SEEDS  = 3
    ABL_MIN_GAIN  = 0.0015
    ABL_MONTH_TOL = 0.003
    ABL_MIN_FOLDS = 3
    ABL_KAPPA_TOL = 0.002
    BLEND_MIN_GAIN = 0.001
    REGIONAL_CAL  = True
    DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'

CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CFG.FIG_DIR.mkdir(parents=True, exist_ok=True)
print(CFG.DATA_DIR, CFG.DEVICE, '| torch', torch.__version__)
if CFG._ON_KAGGLE and CFG.DEVICE != 'cuda':
    raise RuntimeError('GPU not visible to torch (TabPFN and TabM run on the GPU)')

# 3. Data dan preprocessing <a id="data"></a>

Kode pembentukan sampel dan fitur digunakan kembali dari v13. Skala resmi dipertahankan; fitur relative dihitung untuk kesetaraan pipeline, tetapi model memakai fitur absolute B1.

In [ ]:
read = lambda f, **k: pd.read_csv(CFG.DATA_DIR / f, **k)
train   = read('train.csv', parse_dates=['date_show'])
hist    = read('test_history.csv', parse_dates=['date_show'])
test    = read('test.csv', parse_dates=['date_show'])
sub     = read('sample_submission.csv')
movies  = read('movies.csv')
hol     = read('holidays.csv', parse_dates=['date'])
price   = read('ticket_prices.csv')
KEY     = ['movie_title', 'cinema_ids']

for n, x in [('train', train), ('test_history', hist), ('test', test), ('movies', movies),
             ('holidays', hol), ('ticket_prices', price)]:
    print(f'{n:14s} {x.shape}')

In [ ]:
base_title = lambda t: t.str.replace(r'\s*\((IMAX 2D|IMAX 3D|3D)\)\s*$', '', regex=True).str.strip()
fmt = lambda t: t.str.extract(r'\((IMAX 2D|IMAX 3D|3D)\)\s*$')[0].fillna('2D')

d1_test = hist.groupby('movie_title').date_show.min()
h = hist.join(d1_test.rename('d1'), on='movie_title')
h['d'] = (h.date_show - h.d1).dt.days + 1
ph = h.groupby(KEY).d.max().rename('last_day').reset_index()
ph = ph.merge(test[KEY].drop_duplicates().assign(in_test=1), how='left').fillna({'in_test': 0})
display(pd.crosstab(ph.last_day, ph.in_test, margins=True))
nc1 = h[h.d == 1].groupby('movie_title').cinema_ids.nunique()
print('smallest D1 coverage of 2D test films:', nc1[fmt(pd.Series(nc1.index, index=nc1.index)) == '2D'].nsmallest(4).to_dict())
known = set(base_title(pd.Series(train.movie_title.unique()))) | set(base_title(pd.Series(hist.movie_title.unique())))
print('titles in movies.csv used nowhere:', len(set(movies.original_title) - known))
print('D1 weekday of test films:', d1_test.dt.day_name().value_counts().to_dict())

In [ ]:
def release_dates(tx, frac=CFG.D1_FRAC, min_nc=CFG.MIN_NC):
    x = tx.assign(base=base_title(tx.movie_title))
    nc = x.groupby(['base', 'date_show']).cinema_ids.nunique()
    out = {}
    for b, s in nc.groupby(level=0):
        s = s.droplevel(0)
        s = s.reindex(pd.date_range(s.index.min(), s.index.max()), fill_value=0)
        pk = s.idxmax()
        z = s[:pk][s[:pk] == 0]
        seg = s[z.index.max() + pd.Timedelta(days=1):] if len(z) else s
        c = seg[seg >= frac * s.max()]
        if len(c) and c.iloc[0] >= min_nc:
            out[b] = c.index[0]
    t = x.drop_duplicates('movie_title').set_index('movie_title').base
    return t.map(pd.Series(out, dtype='datetime64[ns]')).dropna().rename('d1')


def simulate(tx, d1, last_date=CFG.TRAIN_END, bad=CFG.OUTAGE):
    d1 = d1[d1 + pd.Timedelta(days=9) <= last_date]
    if len(bad):
        d1 = d1[~np.any([(d1 <= b) & (d1 + pd.Timedelta(days=2) >= b) for b in bad], axis=0)]
    x = tx.merge(d1, left_on='movie_title', right_index=True)
    x['d'] = (x.date_show - x.d1).dt.days + 1
    hs = x[x.d.between(1, 3)].drop(columns=['d1', 'd']).reset_index(drop=True)
    pairs = x.loc[x.d == 3, KEY].drop_duplicates().merge(d1, left_on='movie_title', right_index=True)
    tg = pairs.loc[pairs.index.repeat(7)].copy()
    tg['date_show'] = tg.d1 + pd.to_timedelta(np.tile(np.arange(3, 10), len(pairs)), unit='D')
    y = x[x.d.between(4, 10)][KEY + ['date_show', 'total_ticket']]
    tg = tg.merge(y, on=KEY + ['date_show'], how='left').fillna({'total_ticket': 0})
    tg = tg[~tg.date_show.isin(bad)]
    tg['city_name'] = tg.cinema_ids.map(tx.drop_duplicates('cinema_ids').set_index('cinema_ids').city_name)
    return hs, tg.drop(columns='d1').reset_index(drop=True)


def pair_scale(h):
    return (h.groupby(KEY).total_ticket.sum() / 3).clip(lower=1).rename('scale')


def mase(y, p, s):
    return float(np.mean(np.abs(np.asarray(y) - np.asarray(p)) / np.asarray(s)))


first = train.groupby('movie_title').date_show.min()
running = first[first == first.min()].index
D_wide = release_dates(train)
D_lim = release_dates(train, CFG.D1_FRAC, 0).drop(D_wide.index, errors='ignore').drop(running, errors='ignore')
hs, ts = simulate(train, D_wide.drop(running, errors='ignore'))
hl, tl = simulate(train, D_lim)
print(f'wide releases: {ts.movie_title.nunique()} films, {len(ts)} target rows | limited (train only): {tl.movie_title.nunique()} films, {len(tl)} rows')
print('A MINECRAFT MOVIE D1:', D_wide['A MINECRAFT MOVIE'].date())

In [ ]:
dd = pd.date_range('2025-05-01', periods=12)
toy = pd.DataFrame({'date_show': list(dd) + [dd[0], dd[1]], 'cinema_ids': ['A'] * 12 + ['B', 'B'], 'city_name': 'X',
                    'movie_title': 'M', 'total_ticket': list(range(1, 13)) + [5, 5], 'occupation_rate': 1.0,
                    'total_show': 1}).drop(index=5)
th_, tg_ = simulate(toy, pd.Series({'M': dd[0]}, name='d1'), bad=pd.DatetimeIndex([]))
assert set(tg_.cinema_ids) == {'A'} and tg_.total_ticket.tolist() == [4, 5, 0, 7, 8, 9, 10]
assert np.isclose(pair_scale(th_)[('M', 'B')], 10 / 3)
print('simulate() self-check passed')

In [ ]:
def visible_windows(tx, d1):
    x = tx.merge(d1.rename('d1'), left_on='movie_title', right_index=True)
    return x[(x.date_show >= x.d1) & (x.date_show <= x.d1 + pd.Timedelta(days=2))].drop(columns='d1')


vis_tr = visible_windows(train, D_wide)
VIS = pd.concat([vis_tr, hist], ignore_index=True)
VIS['base'] = base_title(VIS.movie_title)
VIS_D1 = VIS.groupby('base').date_show.min().rename('d1')
VP = VIS.groupby(['base', 'cinema_ids']).agg(tix=('total_ticket', 'sum'), shows=('total_show', 'sum')).reset_index().join(VIS_D1, on='base')
VP['sc'], VP['tps'] = VP.tix / 3, VP.tix / VP.shows.clip(lower=1)
VF = (VIS.groupby('base').total_ticket.sum() / 3).rename('ft').to_frame().join(VIS_D1).reset_index()


def ref_cluster(Q, col, win=CFG.REL_WIN, min_n=2):
    q = Q[['movie_title', 'cinema_ids', 'd1']].drop_duplicates()
    q = q.assign(base=base_title(q.movie_title))
    m = q.merge(VP[['base', 'cinema_ids', 'd1', col]].rename(columns={'base': 'base_o', 'd1': 'd1_o'}), on='cinema_ids')
    m = m[((m.d1 - m.d1_o).dt.days.abs() <= win) & (m.base != m.base_o)]
    g = m.groupby(['movie_title', 'cinema_ids'])[col].agg(['median', 'size'])
    med = g['median'].where(g['size'] >= min_n)
    return pd.MultiIndex.from_frame(Q[['movie_title', 'cinema_ids']]).map(med).values.astype(float)


def ref_national(Q, pool, col, win=CFG.REL_WIN):
    fd = Q.drop_duplicates('movie_title').set_index('movie_title').d1
    fb = base_title(pd.Series(fd.index, index=fd.index))
    days = pool.d1.values.astype('datetime64[D]').astype(np.int64)
    out = {}
    for m_, t_ in fd.items():
        k = (np.abs(days - np.datetime64(t_, 'D').astype(np.int64)) <= win) & (pool.base.values != fb[m_])
        out[m_] = np.median(pool[col].values[k])
    return Q.movie_title.map(out).values.astype(float)


In [ ]:
SCHOOL_BREAKS = [('2025-06-28', '2025-07-12'), ('2025-12-22', '2025-12-31')]
JABAR_BREAKS = [('2025-06-30', '2025-07-12'), ('2025-12-29', '2026-01-10')]
JABAR_CITIES = ['BOGOR', 'BEKASI', 'BANDUNG', 'DEPOK', 'CIKARANG', 'KARAWANG', 'CIREBON', 'GARUT', 'TASIKMALAYA',
                'SUMEDANG', 'CIANJUR', 'INDRAMAYU']
CUTI_BERSAMA = pd.to_datetime(['2025-04-02', '2025-04-03', '2025-04-04', '2025-04-07', '2025-05-13', '2025-05-30',
                               '2025-06-09', '2025-08-18', '2025-12-26', '2026-02-16', '2026-03-18', '2026-03-20',
                               '2026-03-23', '2026-03-24'])
RAMADAN = ('2026-02-19', '2026-03-20')


def calendar(hol, breaks=SCHOOL_BREAKS):
    c = hol.rename(columns={'date': 'date_show'})[['date_show', 'holiday_tipe']].copy()
    c['dow'] = c.date_show.dt.dayofweek
    c['is_hol'] = ((c.holiday_tipe == 'holiday') | c.date_show.isin(CUTI_BERSAMA)).astype(int)
    c['school'] = 0
    for a, b in breaks:
        c.loc[c.date_show.between(a, b), 'school'] = 1
    c['ramadan'] = c.date_show.between(*RAMADAN).astype(int)
    base = CFG.DOW_PROF[c.dow]
    base = np.where((c.school == 1) & (c.dow < 4), base * CFG.SCHOOL_WD, base)
    c['cal'] = np.where((c.is_hol == 1) & (c.ramadan == 0), np.maximum(base, CFG.HOL_LEVEL), base)
    return c.drop(columns='holiday_tipe').set_index('date_show')


CAL = calendar(hol)
CAL_JB = calendar(hol, JABAR_BREAKS)
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(CAL.index, CAL.cal, lw=1)
ax.axvline(CFG.TRAIN_END, color=PAL[7], ls='--', label='akhir train')
ax.set(title='Nilai kalender c(t)'); ax.legend()
plt.tight_layout(); plt.savefig(CFG.FIG_DIR / 'cal_value.png'); plt.show()

In [ ]:
GENRES = ['Horror', 'Drama', 'Action', 'Comedy', 'Romance', 'Animation', 'Family', 'Thriller', 'Mystery']
RATING = {'Semua Umur': 0, 'Remaja': 1, 'Dewasa': 2, 'Dewasa 21': 3}


def film_table(vis):
    g = vis.groupby('movie_title')
    return pd.DataFrame({'d1': g.date_show.min(), 'occ': g.occupation_rate.mean(),
                         'tps': g.total_ticket.sum() / g.total_show.sum().clip(lower=1),
                         'logT': np.log1p(g.total_ticket.sum() / 3),
                         'lpc': np.log1p(g.total_ticket.sum() / 3 / g.cinema_ids.nunique().clip(lower=1))})


def make_ctx(visible, size_tx, market):
    rel = visible.assign(base=base_title(visible.movie_title)).groupby('base').agg(
        d1=('date_show', 'min'), tix=('total_ticket', 'sum')).reset_index()
    rel['tix'] /= 3
    cs = size_tx.groupby('cinema_ids').total_ticket.sum() / size_tx.groupby('cinema_ids').date_show.nunique()
    nf = size_tx.groupby(['cinema_ids', 'date_show']).movie_title.nunique().groupby('cinema_ids').mean()
    csh = size_tx.groupby(['cinema_ids', 'date_show']).total_show.sum().groupby('cinema_ids').median()
    return dict(releases=rel, market=market, visible=visible, cin_size=np.log10(cs), cin_nfilms=nf, cin_shows=csh)


def cinema_comp(X, vis, cin_shows):
    v = vis.assign(base=base_title(vis.movie_title))
    agg = v.groupby(['cinema_ids', 'date_show', 'base']).agg(sh=('total_show', 'sum'), tx=('total_ticket', 'sum')).reset_index()
    m = X[['cinema_ids', 'date_show']].assign(base=base_title(X.movie_title)).reset_index().merge(
        agg, on=['cinema_ids', 'date_show'], how='left', suffixes=('', '_o'))
    m = m[m.base != m.base_o]
    g = m.groupby('index').agg(c_new_n=('base_o', 'nunique'), c_new_sh=('sh', 'sum'), c_new_tx=('tx', 'sum'))
    X = X.join(g).fillna({'c_new_n': 0, 'c_new_sh': 0, 'c_new_tx': 0})
    X['c_new_sh_rel'] = X.c_new_sh / X.cinema_ids.map(cin_shows).fillna(X.c_new_sh.median() + 1)
    X['c_new_sh_vs_own'] = X.c_new_sh / X.sh3.clip(lower=1)
    X['c_new_tx_vs_own'] = np.log1p(X.c_new_tx) - np.log1p(X.y3)
    return X


def open_count(X, rel):
    d = np.sort(rel.d1.values.astype('datetime64[D]'))
    a = np.searchsorted(d, X.d1.values.astype('datetime64[D]'), side='right')
    b = np.searchsorted(d, X.date_show.values.astype('datetime64[D]'), side='right')
    return b - a


def build(hh, tgt, ctx):
    mv = movies.set_index('original_title')
    d1 = hh.groupby('movie_title').date_show.min().rename('d1')
    hh = hh.join(d1, on='movie_title')
    hh['d'] = (hh.date_show - hh.d1).dt.days + 1
    piv = lambda v, p: hh.pivot_table(index=KEY, columns='d', values=v, fill_value=0).reindex(columns=[1, 2, 3], fill_value=0).add_prefix(p)
    P = pd.concat([piv('total_ticket', 'y'), piv('total_show', 'sh')], axis=1)
    P['mean3'] = P[['y1', 'y2', 'y3']].mean(axis=1)
    P['scale'] = P.mean3.clip(lower=1)
    P['log_s'] = np.log1p(P.mean3)
    for i in (1, 2, 3):
        P[f'p{i}'] = P[f'y{i}'] / P.scale
    P['n_hist'] = (P[['y1', 'y2', 'y3']] > 0).sum(axis=1)
    P['sh_trend'] = P.sh3 / P[['sh1', 'sh2']].max(axis=1).clip(lower=1)
    P = P.reset_index()

    Fd = hh.groupby(['movie_title', 'd']).agg(T=('total_ticket', 'sum'), nc=('cinema_ids', 'nunique')).unstack().fillna(0)
    Fd.columns = [f'f{a}{b}' for a, b in Fd.columns]
    Fd = Fd.reindex(columns=[f'f{a}{b}' for a in ['T', 'nc'] for b in (1, 2, 3)], fill_value=0)
    Fm = Fd[['fT1', 'fT2', 'fT3']].mean(axis=1).clip(lower=1)
    for i in (1, 2, 3):
        Fd[f'fp{i}'] = Fd[f'fT{i}'] / Fm
    ncmax = Fd[['fnc1', 'fnc2', 'fnc3']].max(axis=1).clip(lower=1)
    Fd['f_logT'] = np.log1p(Fm)
    Fd['f_nc_trend'] = Fd.fnc3 / Fd.fnc1.clip(lower=1)
    Fd['f_occ'] = hh.groupby('movie_title').occupation_rate.mean()
    h3 = hh[hh.d == 3].groupby('movie_title')
    Fd['f_tps3'] = h3.total_ticket.sum() / h3.total_show.sum()
    Fd = Fd.join(d1)
    Fd['base'] = base_title(pd.Series(Fd.index, index=Fd.index))
    Fd['fmt'] = fmt(pd.Series(Fd.index, index=Fd.index)).map({'2D': 0, '3D': 1, 'IMAX 2D': 2, 'IMAX 3D': 3})
    bT = hh.assign(base=base_title(hh.movie_title)).groupby('base').total_ticket.sum() / 3
    Fd['fmt_share'] = np.log1p(Fm) - np.log1p(Fd.base.map(bT))
    Fd['d1_dow'] = Fd.d1.dt.dayofweek
    m = mv.reindex(Fd.base)
    Fd['rating'] = m.age_rating.map(RATING).values
    for g_ in GENRES:
        Fd[f'g_{g_}'] = m.genre.fillna('').str.contains(g_).astype(int).values
    Fd['n_genre'] = m.genre.fillna('').str.count(',').values + 1
    Fd['n_cast'] = m.casts.fillna('').str.count(',').values + 1
    rel = ctx['releases']
    Fd['comp_n'] = [((rel.base != r.base) & (rel.d1 > r.d1) & (rel.d1 <= r.d1 + pd.Timedelta(days=9))).sum() for _, r in Fd.iterrows()]
    Fd['f_share_cohort'] = np.log1p(Fd.base.map(bT)) - np.log1p(Fd.d1.map(rel.groupby('d1').tix.sum()))
    mk = ctx['market']
    win = lambda t: mk[(mk.d1 < t) & (mk.d1 >= t - pd.Timedelta(days=28))]
    Fd[['mk_occ', 'mk_tps', 'mk_logT', 'mk_lpc']] = [win(t)[['occ', 'tps', 'logT', 'lpc']].median().tolist() for t in Fd.d1]
    Fd['occ_rel'] = Fd.f_occ / Fd.mk_occ
    Fd['tps_rel'] = Fd.f_tps3 / Fd.mk_tps
    Fd['logT_rel'] = Fd.f_logT - Fd.mk_logT
    Fd['film_pc_vs_mkt'] = np.log1p(Fm / ncmax) - Fd.mk_lpc

    X = tgt.merge(P, on=KEY, how='left').merge(Fd.drop(columns=['base']), left_on='movie_title', right_index=True, how='left')
    X['h'] = (X.date_show - X.d1).dt.days + 1
    jb = (X.city_name.isin(JABAR_CITIES) & CFG.REGIONAL_CAL).values
    cal_at = lambda dates, col: np.where(jb, CAL_JB[col].reindex(dates).values, CAL[col].reindex(dates).values)
    X['dow'] = X.date_show.dt.dayofweek.values
    X['t_hol'], X['t_school'], X['t_ramadan'] = cal_at(X.date_show, 'is_hol'), cal_at(X.date_show, 'school'), cal_at(X.date_show, 'ramadan')
    ch = np.stack([cal_at(X.d1 + pd.Timedelta(days=k), 'cal') for k in range(3)], 1)
    X['cal_mult'] = cal_at(X.date_show, 'cal') / ch.mean(1)
    X['hol_in_hist'] = np.stack([cal_at(X.d1 + pd.Timedelta(days=k), 'is_hol') for k in range(3)], 1).sum(1)
    # D1-D3 tickets divided by the calendar value of their own day; labels and the MASE scale are untouched
    q = np.stack([X[f'y{i}'].values / cal_at(X.d1 + pd.Timedelta(days=i - 1), 'cal') for i in (1, 2, 3)], 1)
    for i in (1, 2, 3):
        X[f'cal_p{i}'] = q[:, i - 1] / np.maximum(q.mean(1), 1)
    X['cal_log31'] = np.log1p(q[:, 2]) - np.log1p(q[:, 0])
    X['n_comp_open'] = open_count(X, rel)
    X = cinema_comp(X, ctx['visible'], ctx['cin_shows'])
    X['share'] = X.log_s - np.log1p(X[['fT1', 'fT2', 'fT3']].mean(axis=1) / X.fnc3.clip(lower=1))
    X['pair_vs_mkt'] = X.log_s - X.mk_lpc
    X['p3_rel'] = X.p3 - X.fp3
    X['p1_rel'] = X.p1 - X.fp1
    X['cin_size'] = X.cinema_ids.map(ctx['cin_size'])
    X['cin_nfilms'] = X.cinema_ids.map(ctx['cin_nfilms'])
    X['cin_new'] = X.cin_size.isna().astype(int)
    pr = price.pivot(index='city_name', columns='price_day', values='ceil')
    X['price_wkd'] = X.city_name.map(pr.Weekday)
    X['price_prem'] = X.city_name.map(pr.Weekend / pr.Weekday)
    return X

In [ ]:
market = pd.concat([film_table(vis_tr), film_table(hist)])
ctx_tr = make_ctx(vis_tr, train, market)
Xtr = build(hs, ts, ctx_tr)
Xlim = build(hl, tl, ctx_tr)
Xte = build(hist, test, make_ctx(hist, train, market))

assert len(Xtr) == len(ts) and len(Xte) == len(test) and (Xte.id.values == test.id.values).all()
ref = Xte.join(pair_scale(hist), on=KEY, rsuffix='_ref')
assert np.allclose(ref.scale, ref.scale_ref)

BASE_FEATS = ['log_s', 'p1', 'p2', 'p3', 'n_hist', 'sh_trend', 'fnc1', 'fnc2', 'fnc3', 'fp1', 'fp2', 'fp3', 'f_logT',
              'f_nc_trend', 'fmt', 'fmt_share', 'd1_dow', 'rating'] + [f'g_{g_}' for g_ in GENRES] + [
              'n_genre', 'n_cast', 'comp_n', 'f_share_cohort', 'h', 'dow', 't_hol', 't_school', 't_ramadan', 'cal_mult',
              'hol_in_hist', 'n_comp_open', 'share', 'p3_rel', 'p1_rel', 'cin_size', 'cin_nfilms', 'cin_new',
              'price_wkd', 'price_prem']
COMP_FEATS = ['c_new_n', 'c_new_sh', 'c_new_sh_rel', 'c_new_sh_vs_own', 'c_new_tx_vs_own']
CAL_FEATS = ['cal_p1', 'cal_p2', 'cal_p3', 'cal_log31']
FEATS = BASE_FEATS
ZFEATS = FEATS + COMP_FEATS
jb_changed = Xte.t_school.values != CAL.school.reindex(Xte.date_show).values
print(f'West Java calendar: {jb_changed.sum()} test rows change school flag ({jb_changed.mean():.2%}); train rows changed: '
      f'{int((Xtr.t_school.values != CAL.school.reindex(Xtr.date_show).values).sum())}')
for X_ in (Xtr, Xlim, Xte):
    assert np.isfinite(X_[CAL_FEATS].values).all()
print('calendar-normalised inputs (median train | test):', {c: (round(float(Xtr[c].median()), 3), round(float(Xte[c].median()), 3)) for c in CAL_FEATS})
print(f'train {len(Xtr)} rows | limited extra {len(Xlim)} | test {len(Xte)} | features {len(FEATS)}')
display(pd.DataFrame({'train_median': Xtr[['log_s', 'f_logT', 'pair_vs_mkt', 'film_pc_vs_mkt', 'share']].median(),
                      'test_median': Xte[['log_s', 'f_logT', 'pair_vs_mkt', 'film_pc_vs_mkt', 'share']].median()}).round(3))

In [ ]:
REL_FEATS = ['rel_nat', 'rel_clu', 'f_rel']
ABS_LEVEL = ['log_s', 'f_logT']


def add_relative(X):
    X = X.copy()
    X['ref_clu'], X['ref_nat'] = ref_cluster(X, 'sc'), ref_national(X, VP, 'sc')
    ls = np.log(X.scale.values)
    X['rel_nat'] = ls - np.log(np.clip(X.ref_nat.values, 1, None))
    X['rel_clu'] = np.where(np.isfinite(X.ref_clu), ls - np.log(np.clip(X.ref_clu.values, 1, None)), X.rel_nat)
    fm = X[['fT1', 'fT2', 'fT3']].mean(axis=1).clip(lower=1).values
    X['f_rel'] = np.log(fm) - np.log(np.clip(ref_national(X, VF, 'ft'), 1, None))
    return X


Xtr, Xlim, Xte = add_relative(Xtr), add_relative(Xlim), add_relative(Xte)
assert len(Xte) == len(test) and (Xte.id.values == test.id.values).all()
assert all(np.isfinite(X_[REL_FEATS].values).all() for X_ in (Xtr, Xlim, Xte))
FEATS_ABS = BASE_FEATS
FEATS_REL = [f for f in BASE_FEATS if f not in ABS_LEVEL] + REL_FEATS

rng_ = np.random.default_rng(CFG.SEED)
for X_ in (Xtr, Xte):
    for i in rng_.choice(len(X_), 40, replace=False):
        r_ = X_.iloc[i]
        k = (VP.cinema_ids == r_.cinema_ids) & ((VP.d1 - r_.d1).dt.days.abs() <= CFG.REL_WIN) & (VP.base != base_title(pd.Series([r_.movie_title]))[0])
        ref_bf = VP.sc[k].median() if k.sum() >= 2 else np.nan
        assert np.isclose(ref_bf, r_.ref_clu, equal_nan=True), (r_.movie_title, r_.cinema_ids)
print('brute-force check of the cluster reference passed (80 rows); fallback to national reference:',
      {n: round(float(np.isnan(X_.ref_clu).mean()), 4) for n, X_ in [('train', Xtr), ('limited', Xlim), ('test', Xte)]})

kt = pd.DataFrame([dict(feature=c, train_median=Xtr[c].median(), test_median=Xte[c].median(),
                        shift=Xte[c].median() - Xtr[c].median(), ks=ks_2samp(Xtr[c], Xte[c]).statistic)
                   for c in ABS_LEVEL + REL_FEATS]).set_index('feature')
display(kt.round(3))
print(f'cluster reference median (tickets/day): train {np.nanmedian(Xtr.ref_clu):.0f} | test {np.nanmedian(Xte.ref_clu):.0f}')
fig, ax = plt.subplots(1, 4, figsize=(16, 3.4))
for a_, c in zip(ax, ['log_s', 'rel_clu', 'f_logT', 'f_rel']):
    a_.hist(Xtr[c], bins=50, density=True, alpha=.5, label='train'); a_.hist(Xte[c], bins=50, density=True, alpha=.5, label='uji')
    a_.set(title=f"{c}: KS {kt.loc[c, 'ks']:.3f}")
ax[0].legend()
plt.tight_layout(); plt.savefig(CFG.FIG_DIR / 'relative_features.png'); plt.show()

In [ ]:
groups = base_title(Xtr.movie_title).values
ug = np.array(sorted(set(groups)))
fold_of = dict(zip(ug, np.random.RandomState(CFG.SEED).permutation(len(ug)) % CFG.N_FOLDS))
FOLD = np.array([fold_of[g_] for g_ in groups])
y, s, cm, hh_ = Xtr.total_ticket.values, Xtr.scale.values, Xtr.cal_mult.values, Xtr.h.values

bt = pd.cut(Xtr.scale, CFG.TW_BINS)
tw = bt.map(pd.cut(Xte.scale, CFG.TW_BINS).value_counts(normalize=True) / bt.value_counts(normalize=True)).astype(float).values
TW_OLD = tw / tw.mean()
fday = lambda D: pd.Series(np.where(D.y1 > 0, 1, np.where(D.y2 > 0, 2, 3)), index=D.index)
cell = lambda D: pd.cut(D.scale, CFG.TW_BINS).astype(str) + '|' + fday(D).astype(str)
tw = cell(Xtr).map(cell(Xte).value_counts(normalize=True) / cell(Xtr).value_counts(normalize=True)).fillna(0).astype(float).values
TW = tw / tw.mean()
print('late starters (first sale on D3): old weights', round(float(TW_OLD[fday(Xtr) == 3].sum() / len(TW)), 3),
      '| new weights', round(float(TW[fday(Xtr) == 3].sum() / len(TW)), 3), '| test', round(float((fday(Xte) == 3).mean()), 3))

film_ref = pd.Series(Xtr.ref_nat.values).groupby(groups).median()
QUIET_FILMS = set(film_ref[film_ref <= film_ref.quantile(CFG.QUIET_FRAC)].index)
QUIET = np.isin(groups, list(QUIET_FILMS))
print(f'quiet-market lens: {len(QUIET_FILMS)} of {len(film_ref)} films, {QUIET.sum()} rows | national reference median '
      f'busy {np.median(Xtr.ref_nat[~QUIET]):.0f} vs quiet {np.median(Xtr.ref_nat[QUIET]):.0f} vs test {np.median(Xte.ref_nat):.0f} tickets/day')


def score(p, name, yy=None):
    yy = y if yy is None else yy
    e = np.abs(yy - p) / s
    r = dict(model=name, MASE=e.mean(), TW_MASE=np.average(e, weights=TW), TW_OLD=np.average(e, weights=TW_OLD),
             small=e[s <= 20].mean(), big=e[s > 200].mean())
    print(f"{name:40s} MASE {r['MASE']:.4f} | TW-MASE {r['TW_MASE']:.4f} (old {r['TW_OLD']:.4f}) | s<=20 {r['small']:.4f} | s>200 {r['big']:.4f}")
    return r


def train_part(k):
    Xa = Xtr[FOLD != k]
    return pd.concat([Xa, Xlim], ignore_index=True) if CFG.USE_LIMITED else Xa


results = []
rc = y / s / cm
oof_base = np.zeros(len(Xtr))
for k in range(CFG.N_FOLDS):
    a, b = FOLD != k, FOLD == k
    med = pd.Series(rc[a]).groupby([hh_[a], Xtr.d1_dow.values[a]]).median()
    oof_base[b] = med.reindex(pd.MultiIndex.from_arrays([hh_[b], Xtr.d1_dow.values[b]])).fillna(np.median(rc[a])).values * cm[b] * s[b]
results.append(score(np.zeros(len(Xtr)), 'zero'))
results.append(score(oof_base, 'median r/c by (h, D1 dow)'))

In [ ]:
def cin_stats(tx):
    cs = tx.groupby('cinema_ids').total_ticket.sum() / tx.groupby('cinema_ids').date_show.nunique()
    nf = tx.groupby(['cinema_ids', 'date_show']).movie_title.nunique().groupby('cinema_ids').mean()
    csh = tx.groupby(['cinema_ids', 'date_show']).total_show.sum().groupby('cinema_ids').median()
    return np.log10(cs), nf, csh


def with_cin(X, stats):
    cs, nf, csh = stats
    X = X.copy()
    X['cin_size'] = X.cinema_ids.map(cs)
    X['cin_nfilms'] = X.cinema_ids.map(nf)
    X['cin_new'] = X.cin_size.isna().astype(int)
    X['c_new_sh_rel'] = X.c_new_sh / X.cinema_ids.map(csh).fillna(X.c_new_sh.median() + 1)
    return X


TRAIN_BASE = base_title(train.movie_title)
STATS_ALL = cin_stats(train)
assert np.allclose(with_cin(Xtr, STATS_ALL).cin_size.fillna(-9), Xtr.cin_size.fillna(-9))
STATS_FOLD = {k: cin_stats(train[~TRAIN_BASE.isin(set(groups[FOLD == k]))]) for k in range(CFG.N_FOLDS)}
STATS_QUIET = cin_stats(train[~TRAIN_BASE.isin(QUIET_FILMS)])
chg = np.concatenate([(with_cin(Xtr[FOLD == k], STATS_FOLD[k]).cin_size - Xtr.cin_size[FOLD == k]).abs().values for k in range(CFG.N_FOLDS)])
rk = np.mean([pd.Series(with_cin(Xtr[FOLD == k], STATS_FOLD[k]).cin_size.values).corr(pd.Series(Xtr.cin_size[FOLD == k].values), method='spearman') for k in range(CFG.N_FOLDS)])
print(f'|change| of cin_size on validation rows: median {np.nanmedian(chg):.4f}, p95 {np.nanpercentile(chg, 95):.4f} (log10 tickets per day; '
      f'mostly a level shift from removing ~20% of films) | rank correlation global vs fold-local {rk:.4f}')

In [ ]:
LGB_BASE = {k: v for k, v in CFG.LGB_PARAMS.items() if k not in ('objective', 'n_estimators')}
LGB_FAST = {**LGB_BASE, 'n_estimators': 300, 'learning_rate': 0.05}
MONTHS = ['2025-07', '2025-08', '2025-09']
FEATSETS = {'abs': FEATS_ABS, 'rel': FEATS_REL}


def r_target(Xa):
    return (Xa.total_ticket / Xa.scale / Xa.cal_mult).values


def fit_lgb_all(Xa, feats, zfeats):
    l1 = [lgb.LGBMRegressor(objective='l1', n_estimators=CFG.LGB_PARAMS['n_estimators'], **LGB_BASE, random_state=sd)
          .fit(Xa[feats], r_target(Xa), sample_weight=Xa.cal_mult.values) for sd in CFG.SEEDS]
    zc = lgb.LGBMClassifier(objective='binary', **{**LGB_FAST, 'num_leaves': 31}, random_state=CFG.SEED).fit(Xa[zfeats], Xa.total_ticket == 0)
    pos = Xa[Xa.total_ticket > 0]
    qm = [lgb.LGBMRegressor(objective='quantile', alpha=float(q), **LGB_FAST, random_state=CFG.SEED).fit(pos[feats], r_target(pos))
          for q in CFG.QS]
    return dict(l1=l1, zc=zc, qm=qm)


def lgb_size_mb(m):
    return sum(len(zlib.compress(x.booster_.model_to_string().encode(), 9)) for x in m['l1'] + [m['zc']] + m['qm']) / 1e6


def predict_lgb_all(m, Xb, feats, zfeats):
    l1 = np.clip(np.mean([mm.predict(Xb[feats]) for mm in m['l1']], 0), 0, None)
    p0 = m['zc'].predict_proba(Xb[zfeats])[:, 1]
    Q = np.sort(np.column_stack([mm.predict(Xb[feats]) for mm in m['qm']]), axis=1)
    return l1, p0, Q


def mixture_median(p0, Q, qs, lam, a):
    p = sigm(logit(np.clip(p0, 1e-4, 1 - 1e-4)) + lam * a)
    t = np.clip((0.5 - p) / (1 - p), qs[0], qs[-1])
    val = np.array([np.interp(ti, qs, qi) for ti, qi in zip(t, Q)])
    return np.where(p >= 0.5, 0.0, np.clip(val, 0, None))


def lgb_mix(l1, p0, Q, lam=None, a=None):
    lam = CFG.LAMBDA if lam is None else lam
    a = PULL_A if a is None else a
    return (1 - CFG.HURDLE_W) * l1 + CFG.HURDLE_W * mixture_median(p0, Q, CFG.QS, lam, a)


def make_parts(Xa, Xb, cfg, stats):
    if cfg['cin'] == 'local':
        Xa, Xb = with_cin(Xa, stats), with_cin(Xb, stats)
    feats = FEATSETS[cfg['level']]
    return Xa, Xb, feats, feats + COMP_FEATS


def run_cv(cfg):
    oof = dict(l1=np.zeros(len(Xtr)), p0=np.zeros(len(Xtr)), Q=np.zeros((len(Xtr), len(CFG.QS))))
    frames, size = {}, 0.0
    for k in range(CFG.N_FOLDS):
        v = FOLD == k
        Xa, Xb, feats, zfeats = make_parts(train_part(k), Xtr[v], cfg, STATS_FOLD[k])
        m = fit_lgb_all(Xa, feats, zfeats)
        oof['l1'][v], oof['p0'][v], oof['Q'][v] = predict_lgb_all(m, Xb, feats, zfeats)
        frames[k] = (Xa, Xb, feats)
        size = max(size, lgb_size_mb(m))
    return oof, frames, size


def temporal(cfg):
    out = {}
    for mth in MONTHS:
        cut = pd.Timestamp(mth + '-01')
        va = (Xtr.d1.dt.strftime('%Y-%m') == mth).values
        ready = lambda X: X[(X.d1 + pd.Timedelta(days=9) < cut).values]
        Xa = pd.concat([ready(Xtr), ready(Xlim)], ignore_index=True) if CFG.USE_LIMITED else ready(Xtr)
        stats = cin_stats(train[train.date_show < cut]) if cfg['cin'] == 'local' else STATS_ALL
        Xa_, Xb_, feats, zfeats = make_parts(Xa, Xtr[va], cfg, stats)
        out[mth] = dict(idx=np.where(va)[0], parts=predict_lgb_all(fit_lgb_all(Xa_, feats, zfeats), Xb_, feats, zfeats),
                        frames=(Xa_, Xb_, feats), n_train_films=base_title(Xa.movie_title).nunique())
    return out


def quiet_run(cfg):
    Xa = pd.concat([Xtr[~QUIET], Xlim], ignore_index=True) if CFG.USE_LIMITED else Xtr[~QUIET]
    Xa_, Xb_, feats, zfeats = make_parts(Xa, Xtr[QUIET], cfg, STATS_QUIET if cfg['cin'] == 'local' else STATS_ALL)
    return dict(idx=np.where(QUIET)[0], parts=predict_lgb_all(fit_lgb_all(Xa_, feats, zfeats), Xb_, feats, zfeats), frames=(Xa_, Xb_, feats))


def lgb_component(R, lam=None):
    a, o, q = R['a'], R['oof'], R['quiet']
    tp = {m: (t['idx'], lgb_mix(*t['parts'], lam=lam, a=a) * cm[t['idx']] * s[t['idx']]) for m, t in R['temporal'].items()}
    return (lgb_mix(o['l1'], o['p0'], o['Q'], lam=lam, a=a) * cm * s, tp,
            (q['idx'], lgb_mix(*q['parts'], lam=lam, a=a) * cm[q['idx']] * s[q['idx']]))


def lens(p, tpred, qpred):
    e = np.abs(y - p) / s
    r = dict(TW=np.average(e, weights=TW), TW_OLD=np.average(e, weights=TW_OLD), MASE=e.mean())
    r.update({f'fold{k}': np.average(e[FOLD == k], weights=TW[FOLD == k]) for k in range(CFG.N_FOLDS)})
    tm = {m: np.average(np.abs(y[i] - pr) / s[i], weights=TW[i]) for m, (i, pr) in tpred.items()}
    r.update({f'temp_{m}': v for m, v in tm.items()})
    r['temporal_mean'] = np.mean(list(tm.values()))
    qi, qp = qpred
    r['quiet'] = np.average(np.abs(y[qi] - qp) / s[qi], weights=TW[qi])
    r['kappa'] = np.mean([np.average(np.abs(w_ - p) / s, weights=TW) for w_ in WORLDS]) if WORLDS else np.nan
    return r


def blend(ws, comps):
    p = sum(w_ * comps[n][0] for n, w_ in ws.items())
    tp = {m: (i, sum(w_ * comps[n][1][m][1] for n, w_ in ws.items())) for m, (i, _) in comps['lgb'][1].items()}
    qp = (comps['lgb'][2][0], sum(w_ * comps[n][2][1] for n, w_ in ws.items()))
    return p, tp, qp

#### Insights

> Jumlah baris, skala, kalender regional, dan pemeriksaan simulasi harus sama dengan v13 sebelum skor model dibandingkan. Memotong OOF lama berdasarkan bulan tidak menggantikan fitting pada cutoff baru.

# 4. Audit model tetap <a id="audit"></a>

Definisi Fast FP16 dan TabM dipakai kembali. Pergeseran pull dibekukan pada nilai v13; tidak ada grid lambda atau bobot baru. Semua prediksi dan kuantil per split disimpan agar audit berikutnya tidak perlu mengulang fitting.

In [ ]:
logit = lambda p: np.log(p / (1-p))
sigm = lambda x: 1/(1+np.exp(-x))
PULL_A = -1.32
WORLDS = []
TAB_QS = CFG.TAB_QS


def hf_token():
    if os.environ.get('HF_TOKEN'):
        return os.environ['HF_TOKEN']
    if UserSecretsClient is not None:
        try:
            return UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            return None
    return None


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def find_local(fname):
    roots = [Path('/kaggle/input')] if CFG._ON_KAGGLE else []
    for root in roots:
        hit = [p for p in root.rglob(Path(fname).name) if str(p).endswith(fname)]
        if hit:
            return str(hit[0])
        for pk in root.rglob('*.pkl'):
            try:
                with open(pk, 'rb') as f:
                    blob = pickle.load(f).get('fm_files', {})
            except Exception:
                continue
            if fname in blob:
                out = CFG.OUTPUT_DIR / 'fm' / fname
                out.parent.mkdir(parents=True, exist_ok=True)
                out.write_bytes(blob[fname])
                return str(out)
    return None


def checkpoint_path(repo, fname, rev, digest):
    # local_dir keeps the real file name: loaders pick the format from the suffix
    path = find_local(fname) or hf_hub_download(repo, fname, revision=rev, token=hf_token(), local_dir=str(CFG.OUTPUT_DIR / 'fm'))
    got = sha256(path)
    if got != digest:
        raise RuntimeError(f'{fname}: SHA-256 {got} does not match the pinned revision ({digest})')
    print(f'{fname}: SHA-256 verified ({path})')
    return path


def fast16_checkpoint():
    hit = find_local(CFG.FAST16_FILE)
    if hit:
        print(f'{CFG.FAST16_FILE}: found ({hit}), SHA-256 {sha256(hit)}')
        return hit
    src = checkpoint_path(CFG.FAST_REPO, CFG.FAST_FILE, CFG.FAST_REV, CFG.FAST_SHA256)
    out = CFG.OUTPUT_DIR / 'fm' / CFG.FAST16_FILE
    out.parent.mkdir(parents=True, exist_ok=True)
    arrays, err, ref = {}, 0.0, 0.0
    with safe_open(src, framework='np') as f:
        meta = f.metadata()
        for k_ in f.keys():
            a = f.get_tensor(k_)
            b = a.astype(np.float16) if (a.dtype == np.float32 and a.ndim >= 2) else a
            assert np.isfinite(b).all(), k_
            err += float(np.square(b.astype(np.float64) - a).sum())
            ref += float(np.square(a.astype(np.float64)).sum())
            arrays[k_] = b
    save_file(arrays, str(out), metadata=meta)
    Path(src).unlink()
    print(f'{CFG.FAST16_FILE}: {out.stat().st_size / 1e6:.1f} MB, relative weight L2 error {np.sqrt(err / ref):.6f}, SHA-256 {sha256(out)}')
    return str(out)


FM_SPEC = {'tabpfn25q': (ModelVersion.V2_5, CFG.TP25_REPO, CFG.TP25_FILE, CFG.TP25_REV, CFG.TP25_SHA256),
           'tabpfn_v2': (ModelVersion.V2, CFG.TABPFN_REPO, CFG.TABPFN_FILE, CFG.TABPFN_REV, CFG.TABPFN_SHA256),
           'fast35': (ModelVersion.V3_5, CFG.FAST_REPO, CFG.FAST16_FILE, CFG.FAST_REV, CFG.FAST_SHA256)}


def make_tabpfn(version, ckpt):
    def fn(Xa, Xb, feats):
        tf = [c for c in feats if c != 'h']
        out = np.zeros((len(Xb), len(TAB_QS)))
        for hz in range(4, 11):
            ia, ib = (Xa.h == hz).values, (Xb.h == hz).values
            if ib.sum() == 0:
                continue
            m = TabPFNRegressor.create_default_for_version(version, model_path=ckpt, device=CFG.DEVICE,
                                                           n_estimators=CFG.FM_EST, random_state=CFG.SEED, ignore_pretraining_limits=True)
            m.fit(Xa[ia][tf].values.astype(np.float32), r_target(Xa)[ia])
            Qb = Xb[ib][tf].values.astype(np.float32)
            out[ib] = np.concatenate([np.asarray(m.predict(Qb[i:i + 3000], output_type='quantiles', quantiles=list(TAB_QS))).T
                                      for i in range(0, len(Qb), 3000)])
            del m
            torch.cuda.empty_cache()
        return np.sort(out, axis=1)
    return fn


def quant_median(Q, lam, a):
    p0 = np.array([np.interp(CFG.ZERO_EPS, q, TAB_QS, left=0.0, right=1.0) for q in Q])
    p = sigm(logit(np.clip(p0, 1e-4, 1 - 1e-4)) + lam * a)
    u = np.clip(p0 + (1 - p0) * (0.5 - p) / (1 - p), TAB_QS[0], TAB_QS[-1])
    val = np.array([np.interp(ui, TAB_QS, qi) for ui, qi in zip(u, Q)])
    return np.where(p >= 0.5, 0.0, np.clip(val, 0, None))


def run_quant(fn, R):
    oof = np.zeros((len(Xtr), len(TAB_QS)))
    for k in range(CFG.N_FOLDS):
        Xa_, Xb_, f_ = R['frames'][k]
        oof[FOLD == k] = fn(Xa_, Xb_, f_)
    return oof, {m: fn(*t['frames']) for m, t in R['temporal'].items()}, fn(*R['quiet']['frames'])


In [ ]:
TABM_LAST = {}


def pinball(pred, target, w, taus):
    d = target[:, None, None] - pred
    loss = torch.maximum(taus * d, (taus - 1) * d).mean(dim=(1, 2))
    return (loss * w).sum() / w.sum()


def tabm_quantiles(Xa, Xb, feats):
    torch.manual_seed(CFG.SEED)
    np.random.seed(CFG.SEED)
    gA = base_title(Xa.movie_title).values
    ugA = np.array(sorted(set(gA)))
    iv = np.isin(gA, np.random.RandomState(CFG.SEED).permutation(ugA)[: max(1, len(ugA) // 10)])
    # every input statistic is fitted on the inner-training films only (early-stopping films stay unseen)
    fit_part = Xa[~iv]
    use = [f for f in feats if fit_part[f].nunique(dropna=True) > 1]
    med = fit_part[use].median()
    qt = QuantileTransformer(output_distribution='normal', n_quantiles=1000, subsample=10 ** 9, random_state=CFG.SEED)
    qt.fit(fit_part[use].fillna(med).values)
    prep = lambda X: qt.transform(X[use].fillna(med).values).astype(np.float32)
    A, B = prep(Xa), prep(Xb)
    yA, wA = r_target(Xa).astype(np.float32), Xa.cal_mult.values.astype(np.float32)
    dev = torch.device(CFG.DEVICE)
    T = lambda a: torch.as_tensor(a, device=dev)
    Xt, yt, wt, Xv, yv, wv = T(A[~iv]), T(yA[~iv]), T(wA[~iv]), T(A[iv]), T(yA[iv]), T(wA[iv])
    bins = compute_bins(torch.as_tensor(A[~iv]), n_bins=CFG.TABM_BINS)
    model = TabM.make(n_num_features=A.shape[1], d_out=len(TAB_QS), k=CFG.TABM_K,
                      num_embeddings=PiecewiseLinearEmbeddings(bins, CFG.TABM_DEMB, activation=False, version='B')).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.TABM_LR, weight_decay=CFG.TABM_WD)
    taus = T(np.asarray(TAB_QS, dtype=np.float32))
    gen = torch.Generator().manual_seed(CFG.SEED)
    best, best_state, bad = np.inf, None, 0
    for epoch in range(CFG.TABM_EPOCHS):
        model.train()
        perm = torch.randperm(len(Xt), generator=gen).to(dev)
        for i in range(0, len(Xt), CFG.TABM_BATCH):
            b = perm[i:i + CFG.TABM_BATCH]
            loss = pinball(model(Xt[b]), yt[b], wt[b], taus)
            opt.zero_grad()
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            vl = pinball(torch.cat([model(Xv[i:i + 4096]) for i in range(0, len(Xv), 4096)]), yv, wv, taus).item()
        if vl < best - 1e-6:
            best, bad = vl, 0
            best_state = {k_: v_.detach().clone() for k_, v_ in model.state_dict().items()}
        else:
            bad += 1
            if bad >= CFG.TABM_PATIENCE:
                break
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        P = torch.cat([model(T(B[i:i + 4096])).mean(dim=1) for i in range(0, len(B), 4096)]).cpu().numpy()
    TABM_LAST.update(state=best_state, qt=qt, median=med, feats=use, dropped_constant=[f for f in feats if f not in use],
                     bins=bins, epochs=epoch + 1, val_pinball=best)
    return np.sort(P, axis=1)


def tabm_blob():
    buf = io.BytesIO()
    torch.save({k_: v_.cpu() for k_, v_ in TABM_LAST['state'].items()}, buf)
    return {'state': zlib.compress(buf.getvalue(), 9), 'quantile_transformer': pickle.dumps(TABM_LAST['qt']),
            'fill_median': TABM_LAST['median'].to_dict(), 'features': TABM_LAST['feats'],
            'bins': [b_.cpu().numpy() for b_ in TABM_LAST['bins']], 'k': CFG.TABM_K, 'd_embedding': CFG.TABM_DEMB}


In [ ]:
def shifted(p, lam, a):
    p = np.clip(p, 0.0001, 1 - 0.0001)
    return 1 / (1 + np.exp(-(np.log(p / (1 - p)) + lam * a)))

def positive_cdf(values, q, taus):
    j = np.clip((q <= values[:, None]).sum(1) - 1, 0, len(taus) - 2)
    rr = np.arange(len(q))
    lo, hi = (q[rr, j], q[rr, j + 1])
    f = taus[j] + (taus[j + 1] - taus[j]) * np.divide(values - lo, hi - lo, out=np.zeros_like(values), where=hi > lo)
    return np.where(values < q[:, 0], 0, np.where(values >= q[:, -1], 1, np.clip(f, 0, 1)))

def mixture_median(parts, atoms):
    n = len(parts[0][1])
    lo = np.zeros(n)
    hi = np.maximum.reduce([q[:, -1] for _, _, q, _ in parts] + [v for _, v in atoms]) + 1

    def cdf(v):
        return sum((w * (p + (1 - p) * positive_cdf(v, q, taus)) for w, p, q, taus in parts)) + sum((w * (v >= a) for w, a in atoms))
    at_zero = cdf(lo) >= 0.5
    for _ in range(36):
        mid = (lo + hi) / 2
        left = cdf(mid) >= 0.5
        hi = np.where(left, mid, hi)
        lo = np.where(left, lo, mid)
    return np.where(at_zero, 0, hi)

In [ ]:
TARGET_LIKE_FILMS = set(['A BIG BOLD BEAUTIFUL JOURNEY', 'AFTERBURN', 'ANGEL POL', 'ASSALAMUALAIKUM BAITULLAH', 'BAD GENIUS', 'BALLERINA', 'BRIDE HARD', 'DASIM', 'DEMON SLAYER -KIMETSU NO YAIBA- THE MOVIE: INFINITY CASTLE', 'DETECTIVE CONAN THE MOVIE: ONE-EYED FLASHBACK', 'DRACULA: A LOVE TALE', 'DROP', 'FREAKIER FRIDAY', 'GODAAN SETAN YANG TERKUTUK', 'GUNDIK', 'HANYA NAMAMU DALAM DOAKU', 'JADI TUH BARANG', 'JALAN PULANG', 'JODOH 3 BUJANG', 'JURASSIC WORLD: REBIRTH', 'KAMPUNG JABANG MAYIT: RITUAL MAUT', 'LEBIH DARI SELAMANYA', 'LILO & STITCH', 'LOCKED', 'MAMA: PESAN DARI NERAKA', 'MARYAM: JANJI DAN JIWA YANG TERIKAT', 'MATERIALISTS', 'MUNGKIN KITA PERLU WAKTU', 'NOISE', 'PAMALI: TUMBAL', 'PANGGIL AKU AYAH', 'PANJI TENGKORAK', 'PEMBANTAIAN DUKUN SANTET', 'PENCARIAN TERAKHIR', 'PERANG KOTA', 'PEREMPUAN PEMBAWA SIAL', 'RED SONJA', 'REGO NYOWO', 'RELAY', 'SAYAP SAYAP PATAH 2: OLIVIA', 'SERIBU BAYANG PURNAMA', 'SOMEBODY', 'TABAYYUN', 'THARAE: THE EXORCIST', 'THE BAD GUYS 2', 'THE LONG WALK', 'THE NAKED GUN', 'THE SHADOWS EDGE', 'UNTIL DAWN', 'WARFARE', 'WARKOP DKI KARTUN'])
AUDIT_DIR = CFG.OUTPUT_DIR / 'validation_audit'
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
fast_fn = make_tabpfn(ModelVersion.V3_5, fast16_checkpoint())
cases = []
for month in ['2025-06', '2025-07', '2025-08', '2025-09']:
    cutoff = pd.Timestamp(month + '-01')
    mask = Xtr.d1.dt.strftime('%Y-%m').eq(month).values
    ready = lambda X: X[X.d1 + pd.Timedelta(days=9) < cutoff]
    xa = pd.concat([ready(Xtr), ready(Xlim)], ignore_index=True)
    assert (xa.d1 + pd.Timedelta(days=9) < cutoff).all()
    cases.append((month, xa, np.flatnonzero(mask), cin_stats(train[train.date_show < cutoff])))
hold = np.isin(groups, list(TARGET_LIKE_FILMS))
assert len(set(groups[hold])) == 51
extra = Xlim[~base_title(Xlim.movie_title).isin(TARGET_LIKE_FILMS)]
cases.append(('target_like', pd.concat([Xtr[~hold], extra], ignore_index=True), np.flatnonzero(hold),
              cin_stats(train[~TRAIN_BASE.isin(TARGET_LIKE_FILMS)])))
audit_rows = []
for name, xa, idx, stats in cases:
    started = time.time()
    xb = Xtr.iloc[idx]
    assert not set(base_title(xa.movie_title)) & set(base_title(xb.movie_title))
    xa, xb, feats, zfeats = make_parts(xa, xb, dict(cin='local', level='abs'), stats)
    models = fit_lgb_all(xa, feats, zfeats)
    l1, p0, ql = predict_lgb_all(models, xb, feats, zfeats)
    qf, qt = fast_fn(xa, xb, feats), tabm_quantiles(xa, xb, feats)
    c, s_, yy, ww = xb.cal_mult.values, xb.scale.values, xb.total_ticket.values, TW[idx]
    pl = lgb_mix(l1, p0, ql, lam=.5, a=PULL_A)
    pf, pt = quant_median(qf, .5, PULL_A), quant_median(qt, 0., PULL_A)
    part = [(.15, shifted(p0, .5, PULL_A), np.maximum(ql, 0), np.asarray(CFG.QS))]
    taus = np.asarray(TAB_QS)
    for weight, q, lam in [(.5, qf, .5), (.3, qt, 0.)]:
        z = np.array([np.interp(CFG.ZERO_EPS, qr, taus, left=0., right=1.) for qr in q])
        pos = np.maximum(np.array([np.interp(zi+(1-zi)*taus, taus, qr) for zi, qr in zip(z, q)]), 0)
        part.append((weight, shifted(z, lam, PULL_A), pos, taus))
    preds = {'v13_fixed': (.2*pl+.5*pf+.3*pt)*c*s_,
             'without_tabm_fixed': (2/7*pl+5/7*pf)*c*s_,
             'cdf_fixed': mixture_median(part, [(.05, l1)])*c*s_}
    saved_rows = xb[KEY+['h', 'date_show', 'd1', 'scale', 'total_ticket', 'cal_mult']].copy()
    saved_rows['weight'] = ww
    saved_rows['base'] = base_title(xb.movie_title)
    for candidate, pred in preds.items():
        assert np.isfinite(pred).all() and (pred >= 0).all()
        saved_rows[candidate] = pred
        error = np.abs(yy-pred)/s_
        row = dict(split=name, candidate=candidate, train_films=base_title(xa.movie_title).nunique(),
                   val_films=base_title(xb.movie_title).nunique(), rows=len(idx), MASE=error.mean(), TW=np.average(error, weights=ww))
        for seg, mask in {'zero': yy==0, 'positive': yy>0, 'growth': yy>xb.y3.values, 'D4_D5': xb.h.values<=5}.items():
            row[seg] = np.average(error[mask], weights=ww[mask]) if mask.any() else np.nan
        audit_rows.append(row)
    saved_rows.to_csv(AUDIT_DIR / f'{name}_predictions.csv', index=False)
    np.savez_compressed(AUDIT_DIR / f'{name}_quantiles.npz', row_index=idx, l1=l1, p0=p0, lgb_Q=ql, fast_Q=qf, tabm_Q=qt)
    print(name, 'seconds', round(time.time()-started), 'train films', base_title(xa.movie_title).nunique())
    del models
    torch.cuda.empty_cache()
    pd.DataFrame(audit_rows).to_csv(AUDIT_DIR / 'scores.csv', index=False)
scores = pd.DataFrame(audit_rows)
display(scores.round(5))
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
scores.pivot(index='split', columns='candidate', values='TW').plot.bar(ax=ax[0], rot=15)
scores.pivot(index='split', columns='candidate', values='growth').plot.bar(ax=ax[1], rot=15)
ax[0].set_title('Same split: weighted MASE')
ax[1].set_title('Growth errors: watch the trade-off')
plt.tight_layout()
plt.savefig(AUDIT_DIR / 'validation_comparison.png')
plt.show()
print('No submission generated. Compare candidate deltas by split and bootstrap original release cohorts.')

#### Insights

> Kesimpulan diisi setelah output Kaggle tersedia. Jangan menyebut penurunan skor sebagai peningkatan bila hanya berasal dari target nol, satu bulan, atau perubahan populasi evaluasi. Test-like stress split tidak membuktikan label kondisional test serupa. Kandidat baru tetap memerlukan pemeriksaan ukuran paket dan reproduksi sebelum menjadi submission.